# **Lab 5: Frequency response of FIR filters**

The goal of this lab is to connect the time-domain ideas from the earlier labs with the frequency-domain view of FIR filters. In this notebook we will study three related ideas:

- how a moving-average filter smooths a signal,
- how the frequency response of an FIR filter shows which frequencies are passed or attenuated, and
- how a nulling filter can remove a single interfering sinusoid.

These ideas are the core of the theory summary on frequency response of FIR filters.

Recall that an FIR filter is defined by the convolution

$y[n] = \sum_{k=0}^{N-1} b_k x[n-k]$

and that its frequency response is obtained from

$H(e^{j\omega}) = \sum_{k=0}^{N-1} b_k e^{-j\omega k}$.

In particular, a moving-average filter with coefficients $b_k = 1/N$ has a response that is largest at low frequencies and decreases as the frequency increases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from IPython.display import Audio, display

from util import load_audio, plot_signals, plot_spectrogram, plot_frequency_response

## 1. Reuse the harmonic synthesis from Lab 4

We will first recreate a simple harmonic sound, then we will study how FIR filters change its shape in the time and frequency domains.

In [ ]:
def synthesize(f0, phi, Ak, t):
    y = np.zeros_like(t, dtype=float)
    for k in range(1, len(Ak) + 1):
        y += Ak[k - 1] * np.cos(2 * np.pi * k * f0 * t + k * phi - (k - 1) * np.pi / 2)
    return y

reference_audio, fs = load_audio('audio/reference.wav')
t = np.arange(len(reference_audio)) / fs

weights = [0.8, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05]
harmonic_signal = synthesize(196.0, 0.0, weights, t)
harmonic_signal = harmonic_signal / np.max(np.abs(harmonic_signal))

plot_signals(harmonic_signal, fs, t_start=0, t_end=0.02, name='harmonic signal')

### 1.1 Questions

1. What features of the synthesized signal are similar to the reference audio?
   - Write your answer here.

2. What features are still different from the reference audio?
   - Write your answer here.

3. Why do you think a simple harmonic model is useful even though it is not identical to the real sound?
   - Write your answer here.

## 2. Frequency response of an averaging filter

A moving-average filter is an FIR filter with coefficients that are all equal. Its frequency response is easy to compute and reveals the filter’s low-pass behavior. A good filter is characterized by its magnitude response, which tells us which frequencies are passed, and its phase response, which tells us how the filter shifts the input components.

In [ ]:
N = 4
b = np.ones(N) / N
plot_frequency_response(b, sr=fs)

In [ ]:
for N in [2, 4, 8]:
    b = np.ones(N) / N
    print(f'--- Moving average with N={N} ---')
    plot_frequency_response(b, sr=fs)
    plt.show()

### 2.1 Questions

1. What happens to the magnitude response when $N$ becomes larger?
   - Write your answer here.

2. Which frequencies are attenuated more strongly by the moving-average filter: low frequencies or high frequencies?
   - Write your answer here.

3. Why does the moving-average filter act as a low-pass filter?
   - Write your answer here.

## 3. Nulling filters for rejection

A nulling filter is an FIR filter that has a zero in its frequency response at one chosen frequency. This makes it useful for removing a single sinusoidal interference component. The filter

$y[n] = x[n] - 2\cos(\omega_n) x[n-1] + x[n-2]$

has a null at $\omega = \omega_n$. In this section we will use it to remove a 1000 Hz interference from the reference signal.

In [ ]:
def add_interference(x, fs, f_interf=1000.0, amplitude=0.25):
    t = np.arange(len(x)) / fs
    interference = amplitude * np.sin(2 * np.pi * f_interf * t)
    return x + interference

x_interf = add_interference(reference_audio, fs)

plot_signals([reference_audio, x_interf], fs, t_start=0, t_end=0.02, name=['reference', 'corrupted'])

In [ ]:
ax1 = plot_spectrogram(reference_audio, sr=fs, N=2048, H=256)
ax2 = plot_spectrogram(x_interf, sr=fs, N=2048, H=256)

ax1.set_ylim([0, 4000])
ax2.set_ylim([0, 4000])
ax1.set_title('reference spectrogram')
ax2.set_title('corrupted spectrogram')
plt.show()

### 3.1 Questions

1. What changes do you see in the time-domain plot after adding the interference?
   - Write your answer here.

2. What extra structure appears in the spectrogram after the interference is added?
   - Write your answer here.

3. Why does a nulling filter work well for this kind of interference?
   - Write your answer here.

In [ ]:
def remove_interference(x, fs, f0=1000.0):
    omega_n = 2 * np.pi * f0 / fs
    b = np.array([1.0, -2 * np.cos(omega_n), 1.0])
    return np.convolve(x, b, mode='same')

x_clean = remove_interference(x_interf, fs)

ax = plot_spectrogram(x_clean, sr=fs, N=2048, H=256)
ax.set_ylim([0, 4000])
ax.set_title('cleaned spectrogram')
plt.show()

### 3.2 Final reflection

Compare the original, corrupted, and cleaned signals. Listen to the sounds carefully and describe what changed after adding and removing the interference.

1. What do you hear before and after the interference is added?
   - Write your answer here.

2. What do you hear after the nulling filter is applied?
   - Write your answer here.

3. How does the frequency-response plot help explain what you heard?
   - Write your answer here.